# 01 - RAG retrieval, step by step

**Goal:** see the RAG pipeline of the project in the order of its files (`src/rag/`, one file per
step), then follow the course labs 10 to 16 on the project's index: the Qdrant collection, one
question searched in dense, BM25 and hybrid mode, with query rewriting and with reranking.
Everything goes through `src/rag/` (the code the agent uses); nothing is written to Qdrant.
Needs Qdrant and Ollama running (`docker compose up -d`).

In [1]:
import os
import sys
import warnings
from pathlib import Path

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
ROOT = Path.cwd().resolve().parents[1]  # notebooks/<module>/ -> repository root
for path in (ROOT, ROOT / "backend"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
os.chdir(ROOT)
warnings.filterwarnings("ignore")

import logging
import tempfile

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from PIL import Image

plt.rcParams.update({"figure.dpi": 72, "savefig.dpi": 72})
%config InlineBackend.figure_formats = ["jpeg"]
pd.set_option("display.max_colwidth", 90)


def fmt(frame, digits=3, **columns):
    """Format the float columns with f"{x:.{digits}f}", the rounding of the result tables."""
    out = frame.copy()
    for name in out.columns:
        if out[name].dtype.kind == "f":
            d = columns.get(name, digits)
            out[name] = out[name].map(lambda v, d=d: "" if pd.isna(v) else f"{v:.{d}f}")
    return out


def show_image(path, title="", width=480):
    """Display an image file, downscaled (the notebook stays small)."""
    import io

    from IPython.display import Image as JpegImage

    image = Image.open(path).convert("RGB")
    image.thumbnail((width, width))
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=80)
    if title:
        display(Markdown(f"*{title}*"))
    display(JpegImage(data=buffer.getvalue(), format="jpeg"))

## 0. The pipeline, one file per step

```text
Sources: openFDA drug labels (JSON) · PubMed abstracts (XML) · DDInter interactions (CSV)
   |
[step1 — Extractor]   connectors -> clean text per document          src/rag/extractor.py
[step2 — Metadata]    ingredient · brand · source · section ·        src/rag/metadata.py
                       language · year · url
[step3 — Chunker]     ~1500-char chunks with overlap,                src/rag/chunking.py
                       label sections kept whole
[step4 — Embedder]    BAAI/bge-m3 via Ollama (1024-dim) + BM25        src/rag/embedder.py
[step5 — Qdrant]      tujya_chunks_v2, dense HNSW cosine + sparse,    src/rag/vector_store.py
                       add / update / delete a document  <---------+
User question -> drug-name detection (brand -> ingredient)          |  src/rag/query_rewriting.py
   + query rewriting -> hybrid search (dense + BM25, RRF), top-k ---+  src/rag/retrieval.py
   (reranker measured, not served)                                     src/rag/reranking.py
[step6 — LLM]         qwen2.5:3b via Ollama + prompt v1               src/rag/rag_chain.py
   -> safety check (no dose, no "safe", cited sentences only)          src/agent/safety.py
   -> Answer + citations + disclaimer
```

| Step | File | Command |
|---|---|---|
| 1 Extractor | `src/rag/extractor.py` | `python -m src.rag.extractor labels`, `pubmed`, `ddinter`, `interactions` |
| 2 Metadata | `src/rag/metadata.py` | `python -m src.rag.extractor ingredients` |
| 3 Chunker | `src/rag/chunking.py` | `dvc repro spark_rag` |
| 4 Embedder | `src/rag/embedder.py` | called by step 5 |
| 5 Qdrant | `src/rag/vector_store.py` | `python -m src.rag.vector_store index`; `add`, `update`, `delete` |
| Query | `query_rewriting.py`, `retrieval.py`, `reranking.py` | `python -m src.rag.retrieval "..."` |
| 6 LLM | `src/rag/rag_chain.py` | `python -m src.agent.ask "..."` |

Each file starts with the step it implements:

In [2]:
import ast

ORDER = [
    "extractor",
    "metadata",
    "chunking",
    "embedder",
    "vector_store",
    "query_rewriting",
    "retrieval",
    "reranking",
    "rag_chain",
]
rows = []
for name in ORDER:
    path = ROOT / "src" / "rag" / f"{name}.py"
    docstring = ast.get_docstring(ast.parse(path.read_text(encoding="utf-8"))) or ""
    rows.append(
        {
            "file": f"src/rag/{name}.py",
            "lines": len(path.read_text(encoding="utf-8").splitlines()),
            "step": docstring.splitlines()[0],
        }
    )
pd.set_option("display.max_colwidth", 110)
display(pd.DataFrame(rows))
pd.set_option("display.max_colwidth", 90)

,file,lines,step
0,src/rag/extractor.py,278,"Step 1 — Extractor: source documents (openFDA labels, PubMed abstracts, DDInter) -> clean text."
1,src/rag/metadata.py,218,Step 2 — Metadata: the names that tie every document to the pill classes.
2,src/rag/chunking.py,415,"Step 3 — Chunker: openFDA label sections, PubMed abstracts and DDInter rows -> chunks."
3,src/rag/embedder.py,220,Step 4 — Embedder: each chunk becomes a dense bge-m3 vector and a sparse BM25 vector.
4,src/rag/vector_store.py,1190,"Step 5 — Vector store: the chunks, their metadata and their vectors in Qdrant."
5,src/rag/query_rewriting.py,128,"Query path — Drug-name detection and query rewriting, before retrieval."
6,src/rag/retrieval.py,470,"Query path — Retrieval: BM25, dense (bge-m3) or hybrid (reciprocal rank fusion), top k."
7,src/rag/reranking.py,48,"Query path — Reranking: a cross-encoder reorders the retrieved candidates (measured, not served)."
8,src/rag/rag_chain.py,94,Step 6 — LLM: question -> search -> numbered sources -> prompt v1 -> qwen2.5:3b -> cited answer.


## 1. The collection (lab 10)

One point per chunk: a dense bge-m3 vector, a BM25 sparse vector and the chunk's metadata.

In [3]:
from qdrant_client import QdrantClient

from app.config import Settings

settings = Settings()
qdrant = QdrantClient(url=settings.qdrant_url)
info = qdrant.get_collection(settings.qdrant_collection)
print(f"collection {settings.qdrant_collection}: {info.points_count} points")
print(
    "dense vector:", {k: (v.size, str(v.distance)) for k, v in info.config.params.vectors.items()}
)
print("sparse vectors:", list(info.config.params.sparse_vectors))
point = qdrant.scroll(settings.qdrant_collection, limit=1, with_payload=True)[0][0]
pd.Series(
    {k: str(v)[:80] for k, v in point.payload.items()}, name="payload of one point"
).to_frame()

collection tujya_chunks_v2: 27302 points
dense vector: {'dense': (1024, 'Cosine')}
sparse vectors: ['bm25']


,payload of one point
chunk_id,593aee3a2ba905c8
doc_id,ddinter:DDInter771:DDInter1662
doc_type,interaction
ingredients,"['folic acid', 'sertraline']"
mentioned_ingredients,[]
section,interaction
chunk_index,0
title,DDInter 2.0: folic acid + sertraline (Unknown)
text,DDInter 2.0 lists an interaction whose severity DDInter does not grade (level Un
source_name,DDInter 2.0


## 2. Retriever (lab 11)

The agent's retriever: Qdrant store, bge-m3 embeddings from Ollama, BM25 sparse encoder and
cross-encoder reranker. The fastembed models are cached in a temporary folder.

In [4]:
from src.rag.embedder import FastEmbedBM25, OllamaEmbedder
from src.rag.reranking import CrossEncoderReranker
from src.rag.retrieval import Retriever
from src.rag.vector_store import Filters, open_store

cache = Path(tempfile.gettempdir()) / "tujya-fastembed"
for name in ("httpx", "httpcore", "huggingface_hub"):
    logging.getLogger(name).setLevel(logging.WARNING)
embedder = OllamaEmbedder(
    settings.ollama_url,
    settings.embedding_model,
    settings.embedding_dim,
    num_ctx=settings.embedding_num_ctx,
)
retriever = Retriever(
    open_store(settings, embedder.dim),
    embedder,
    sparse=FastEmbedBM25(settings.rag_sparse_model, cache_dir=cache),
    reranker=CrossEncoderReranker(settings.rag_rerank_model, cache_dir=cache),
)
print(f"{len(retriever.chunks)} chunks loaded, BM25 runs in: {retriever.lexical_kind}")

27302 chunks loaded, BM25 runs in: qdrant-sparse:Qdrant/bm25


## 3. One question, three modes (labs 14 and 15)

The ingredients named in the question become a metadata filter, as the agent does.

In [5]:
QUESTION = "La warfarine interagit-elle avec le fluconazole ?"
FILTERS = Filters(ingredients=("warfarin", "fluconazole"))


def table(hits):
    return pd.DataFrame(
        [
            {"score": round(h.score, 4), "type": h.chunk.doc_type, "title": h.chunk.title[:70]}
            for h in hits
        ]
    )


async def search(query, mode, rerank=False):
    return await retriever.search(query, k=3, mode=mode, filters=FILTERS, rerank=rerank)


for mode in ("dense", "bm25", "hybrid"):
    display(Markdown(f"**{mode}**"))
    display(table(await search(QUESTION, mode)))

**dense**

,score,type,title
0,0.0164,label,Fluconazole label (Fluconazole): Drug interactions
1,0.0161,label,Warfarin label (Warfarin Sodium): Drug interactions
2,0.0159,label,Fluconazole label (Fluconazole): Precautions


**bm25**

,score,type,title
0,0.0164,interaction,DDInter 2.0: fluconazole + warfarin (Major)
1,0.0161,label,Fluconazole label (Fluconazole): Precautions
2,0.0159,label,Fluconazole label (Fluconazole): Drug interactions


**hybrid**

,score,type,title
0,0.0323,label,Fluconazole label (Fluconazole): Drug interactions
1,0.0320,interaction,DDInter 2.0: fluconazole + warfarin (Major)
2,0.0320,label,Fluconazole label (Fluconazole): Precautions


## 4. Query rewriting (lab 13)

The corpus is in English and BM25 only matches English words: the LLM rewrites the French
question into a short English query (`src/rag/query_rewriting.py`).

In [6]:
from src.agent.runtime import ollama_llm
from src.rag.query_rewriting import QueryRewriter

rewritten = await QueryRewriter(ollama_llm(settings)).rewrite(QUESTION)
print("rewritten query:", rewritten)
display(table(await search(rewritten, "hybrid")))

rewritten query: warfarine interacts fluconazole PubMed


,score,type,title
0,0.0325,label,Fluconazole label (Fluconazole): Drug interactions
1,0.0323,interaction,DDInter 2.0: fluconazole + warfarin (Major)
2,0.0320,label,Fluconazole label (Fluconazole): Precautions


## 5. Reranking (lab 16)

A cross-encoder reads the question with each candidate and reorders them. It is more precise
but costs seconds per question on the CPU, so the app keeps it off by default.

In [7]:
display(table(await search(rewritten, "hybrid", rerank=True)))

,score,type,title
0,5.0827,label,Warfarin label (Warfarin Sodium): Drug interactions
1,5.0646,label,Fluconazole label (Fluconazole): Drug interactions
2,4.3695,interaction,DDInter 2.0: fluconazole + warfarin (Major)
